# Practica de Modelos Lineales y Gradiente por Descendiente, con y sin Regularización

### Importaciones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Generar datos sintéticos

np.random.seed(42)

X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

X_b = np.c_[np.ones((100, 1)), X]  # add x0 = 1 to each instance

## Veamos como podemos utilizar el Gradiente por Descenciente en Sckit-Learn

In [ ]:
# Importar librerías de Scikit-Learn
from sklearn.linear_model import SGDRegressor

# Crear el modelo de regresión lineal con descenso por gradiente
sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3 , penalty=None, eta0=0.1)    # penalty=None indica que no usamos regularización, el resto de parámetros son para controlar la convergencia y eta0 es la tasa de aprendizaje, max_iter es el número máximo de iteraciones
sgd_reg.fit(X, y.ravel())  # Ajustar el modelo a los datos
print("Pendiente (w1):", sgd_reg.coef_[0])
print("Bias (w0):", sgd_reg.intercept_[0])


In [ ]:
# Diferentes funciones de coste
def plot_cost_function(X, y, w0_range, w1_range):
    W0, W1 = np.meshgrid(w0_range, w1_range)
    Z = np.zeros(W0.shape)

    for i in range(W0.shape[0]):
        for j in range(W0.shape[1]):
            w = np.array([W0[i, j], W1[i, j]])
            Z[i, j] = np.mean((y - X_b.dot(w.reshape(-1, 1))) ** 2)

    plt.figure(figsize=(10, 6))
    cp = plt.contourf(W0, W1, Z, levels=50, cmap='viridis')
    plt.colorbar(cp)
    plt.xlabel('w0 (Bias)')
    plt.ylabel('w1 (Pendiente)')
    plt.title('Función de coste (MSE)')
    plt.scatter(sgd_reg.intercept_[0], sgd_reg.coef_[0], color='red', marker='x', s=100, label='Mínimo encontrado')
    plt.legend()
    plt.show()

    # Graficar la función de coste
plot_cost_function(X_b, y, w0_range=np.linspace(-1, 5, 100), w1_range=np.linspace(2, 4, 100))
# Predicciones
X_new = np.array([[0], [2]])
y_predict = sgd_reg.predict(X_new)
print("Predicciones para X_new = [[0], [2]]:", y_predict)
# Visualización de los resultados
plt.scatter(X, y, color='blue', label='Datos de entrenamiento')
plt.plot(X_new, y_predict, color='red', linewidth=2, label='Modelo SGD')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Regresión Lineal con Descenso por Gradiente (SGD)')
plt.legend()
plt.show()  

### Utilización de diferentes tasas de aprendizaje


In [ ]:
# Utilizar diferentes tasas de aprendizaje
etas = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0]
plt.figure(figsize=(12, 8))
for i, eta in enumerate(etas):
    sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None, eta0=eta)
    sgd_reg.fit(X, y.ravel())
    y_predict = sgd_reg.predict(X_new)
    
    plt.subplot(2, 3, i+1)
    plt.scatter(X, y, color='blue', label='Datos de entrenamiento')
    plt.plot(X_new, y_predict, color='red', linewidth=2, label=f'Modelo SGD (eta={eta})')
    plt.xlabel('X')
    plt.ylabel('y')
    plt.title(f'Regresión Lineal con eta={eta}')
    plt.legend()
plt.tight_layout()
plt.show()

## Características Polinómicas

In [ ]:
# Generar datos no lineales polinómicos utilizar sklearn


np.random.seed(42)
X = 2 * np.random.rand(100, 1) - 1  # Datos en el rango [-1, 1]
y = 0.5 * X**3 - X**2 + X + np.random.randn(100, 1) * 0.1  # Relación polinómica con ruido

# Visualizar los datos
plt.scatter(X, y)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos Sintéticos No Lineales')
plt.show()
# Transformar las características a polinómicas
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)
print("Características originales:\n", X[:5])
print("Características polinómicas:\n", X_poly[:5])
# Ajustar el modelo de regresión lineal con descenso por gradiente a las características polinómicas
sgd_reg_poly = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None, eta0=0.1)
sgd_reg_poly.fit(X_poly, y.ravel())
print("Coeficientes del modelo polinómico:", sgd_reg_poly.coef_)
print("Intercepto del modelo polinómico:", sgd_reg_poly.intercept_)
# Predicciones con el modelo polinómico
X_new = np.linspace(-1, 1, 100).reshape(100, 1)
X_new_poly = poly.transform(X_new)
y_new_predict = sgd_reg_poly.predict(X_new_poly)
# Visualización de los resultados
plt.scatter(X, y, color='blue', label='Datos de entrenamiento')
plt.plot(X_new, y_new_predict, color='red', linewidth=2, label='        Modelo Polinómico SGD')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Regresión Polinómica con Descenso por Gradiente (SGD)')
plt.legend()
plt.show()

## Regularización en Sckit-Learn

### Sin gradiente por descendiente. Mínimos cuadrados

In [ ]:
# Ridge y Lasso con Scikit-Learn

# Generamos datos lineales multivariables con ruido y con diferentes escalas con tres variables
np.random.seed(42)
X = np.random.rand(1000, 3) * np.array([1, 10, 100])  # Diferentes escalas
y = 3 + 1 * X[:, 0] + 0.5 * X[:, 1] + 0.1 * X[:, 2] + np.random.randn(1000) * 0.5  # Relación lineal con ruido

# Divide los datos en train y test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#Visualizar los datos
print("Primeras 5 filas de X:\n", X[:5])
print("Primeras 5 filas de y:\n", y[:5])

# Estandarizar las características del train y test
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Ajustar modelos de regresión Ridge y Lasso para el train
from sklearn.linear_model import Ridge, Lasso
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

print("Coeficientes Ridge:", ridge.coef_)
print("Intercepto Ridge:", ridge.intercept_)
print("Coeficientes Lasso:", lasso.coef_)
print("Intercepto Lasso:", lasso.intercept_)

# Predicciones con ambos modelos para el test
y_ridge_predict = ridge.predict(X_test)
y_lasso_predict = lasso.predict(X_test)

# Medición del error cuadrático medio (MSE) para ambos modelos
from sklearn.metrics import mean_squared_error

mse_ridge = mean_squared_error(y_test, y_ridge_predict)
mse_lasso = mean_squared_error(y_test, y_lasso_predict)

print("Error Cuadrático Medio (MSE) Ridge:", mse_ridge)
print("Error Cuadrático Medio (MSE) Lasso:", mse_lasso)

# Visualización de los resultados
plt.figure(figsize=(12, 6))
plt.scatter(y_test, y_ridge_predict, color='blue', label='Ridge Predictions')
plt.scatter(y_test, y_lasso_predict, color='red', label='Lasso Predictions')
plt.xlabel("Valores Reales")
plt.ylabel("Predicciones")
plt.title("Comparación de Predicciones: Ridge vs Lasso")
plt.legend()
plt.show()

In [ ]:
# Para los mismos datos, ajustar mediante SGDRegressor sin regularización, con Ridge y con Lasso para los datos de test y train
sgd_no_reg = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None, eta0=0.1)
sgd_ridge = SGDRegressor(max_iter=1000, tol=1e-3, penalty='l2', alpha=1.0, eta0=0.1) 
sgd_lasso = SGDRegressor(max_iter=1000, tol=1e-3, penalty='l1', alpha=0.1, eta0=0.1)
sgd_no_reg.fit(X_train, y_train)
sgd_ridge.fit(X_train, y_train)
sgd_lasso.fit(X_train, y_train)
print("Coeficientes SGD sin regularización:", sgd_no_reg.coef_)
print("Intercepto SGD sin regularización:", sgd_no_reg.intercept_)
print("Coeficientes SGD con Ridge:", sgd_ridge.coef_)
print("Intercepto SGD con Ridge:", sgd_ridge.intercept_)
print("Coeficientes SGD con Lasso:", sgd_lasso.coef_)
print("Intercepto SGD con Lasso:", sgd_lasso.intercept_)
y_sgd_no_reg_predict = sgd_no_reg.predict(X_test)
y_sgd_ridge_predict = sgd_ridge.predict(X_test)
y_sgd_lasso_predict = sgd_lasso.predict(X_test)
plt.figure(figsize=(12, 6))
plt.scatter(y_test, y_sgd_no_reg_predict, color='green', label='SGD No Reg')
plt.scatter(y_test, y_sgd_ridge_predict, color='blue', label='SGD Ridge')
plt.scatter(y_test, y_sgd_lasso_predict, color='red', label='SGD Lasso')
plt.xlabel("Valores Reales")    
plt.ylabel("Predicciones")
plt.title("Comparación de Predicciones: SGD No Reg vs Ridge vs Lasso")
plt.legend()
plt.show()

# Medición del error cuadrático medio (MSE) para los tres modelos SGD
mse_sgd_no_reg = mean_squared_error(y_test, y_sgd_no_reg_predict)
mse_sgd_ridge = mean_squared_error(y_test, y_sgd_ridge_predict)
mse_sgd_lasso = mean_squared_error(y_test, y_sgd_lasso_predict)
print("Error Cuadrático Medio (MSE) SGD sin regularización:", mse_sgd_no_reg)
print("Error Cuadrático Medio (MSE) SGD con Ridge:", mse_sgd_ridge)
print("Error Cuadrático Medio (MSE) SGD con Lasso:", mse_sgd_lasso)



# CURVA DE APRENDIZAJE

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# Generar datos sintéticos
np.random.seed(42)
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.randn(m, 1) * 0.2
# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear un conjunto de datos nuevo para predicciones
X_new = np.linspace(-3, 3, 100).reshape(100, 1)
# Usar un pipeline para ajunstar modelos de regresion polinomica de diferentes grados
plt.figure(figsize=(10, 6))

for style, width, degree in (("g-", 1, 100), ("b--", 2, 2), ("o-", 2, 1)):
    polybig_features = PolynomialFeatures(degree=degree, include_bias=False)
    std_scaler = StandardScaler()
    lin_reg = LinearRegression()
    polynomial_regression = Pipeline([
            ("poly_features", polybig_features),
            ("std_scaler", std_scaler),
            ("lin_reg", lin_reg),
        ])
    polynomial_regression.fit(X_train, y_train)
    y_newbig = polynomial_regression.predict(X_new)
    plt.plot(X_new, y_newbig, style, label=str(degree), linewidth=width)

plt.plot(X_train, y_train, "b.", linewidth=3)
plt.plot(X_test, y_test, "r.", linewidth=3)
plt.title("Regresión Polinómica de Diferentes Grados", fontsize=18)
plt.legend(loc="upper left")
plt.xlabel("$x_1$", fontsize=18)
plt.ylabel("$y$", rotation=0, fontsize=18)
plt.axis([-3, 3, 0, 10])
plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

def plot_learning_curves(model, X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)
    train_errors, val_errors = [], []
    for m in range(1, len(X_train)):
        model.fit(X_train[:m], y_train[:m])
        y_train_predict = model.predict(X_train[:m])
        y_test_predict = model.predict(X_test)
        train_errors.append(mean_squared_error(y_train[:m], y_train_predict))
        val_errors.append(mean_squared_error(y_test, y_test_predict))

    plt.plot(np.sqrt(train_errors), "r-", linewidth=2, label="train")
    plt.plot(np.sqrt(val_errors), "b-", linewidth=3, label="test")
    plt.legend(loc="upper right", fontsize=14)   
    plt.xlabel("Training set size", fontsize=14) 
    plt.ylabel("RMSE", fontsize=14)    

lin_reg = LinearRegression()
plot_learning_curves(lin_reg, X, y)
plt.axis([0, 80, 0, 3])                      
plt.show()     


# EjERCICIOS


## EJERCICIO 1 — `gradient_descent(X, y, learning_rate, n_iterations, w_init)` (autocorregible)

Implementa la función **Gradient_descent** (Gradiente por descendiente). Debe devolver un vector de pesos.


In [ ]:
# EJE1
### Gradiente por descendiente. Función para Regresión Lineal sin regularizacion
def gradient_descent(X, y, learning_rate=0.1, n_iterations=1000, w_init=None):
    raise NotImplementedError("Implementar la función de gradiente por descenso")

In [ ]:
# TEST UNITARIO
try:
    import numpy as np

    # Datos sencillos: y = x (pendiente=1, bias=0)
    X_test_unit = np.array([[1], [2], [3]])
    y_test_unit = np.array([1, 2, 3])

    # Ejecutar la función
    # iniciar semilla y pesos iniciales
    np.random.seed(42)
    initial_w = np.random.randn(2)  # Tamaño 2 para bias + pendiente
    w_test_unit = gradient_descent(X_test_unit, y_test_unit, learning_rate=0.1, n_iterations=1000, w_init=initial_w)

    print("Pesos aprendidos (test unitario):", w_test_unit)

    # Verificaciones
    assert w_test_unit.shape[0] == 2, "❌ w debe tener tamaño 2 (bias + pendiente)"
    assert np.allclose(w_test_unit, [0, 1], atol=0.1), f"❌ w no es correcto, obtenido {w_test_unit}"

    print("✅ Test EJ1 superado")

except NotImplementedError:
    print("❌ EJ1 pendiente: implementa gradient_descent")

except AssertionError as e:
    print(str(e))

except Exception as e:
    print(f"⚠️ Error inesperado en test EJ1: {e}")


In [ ]:
# TEST UNITARIO comparando con scikit-learn
try:
    import numpy as np
    from sklearn.linear_model import LinearRegression

    # Datos sencillos: y = x (pendiente=1, bias=0)
    X_test_unit = np.array([[1], [2], [3]])
    y_test_unit = np.array([1, 2, 3])

    # Nuestra implementación
    w_custom = gradient_descent(X_test_unit, y_test_unit, learning_rate=0.1, n_iterations=5000, w_init=np.array([0.0, 0.0]))

    # Implementación scikit-learn
    lin_reg = LinearRegression()
    lin_reg.fit(X_test_unit, y_test_unit)
    w_sklearn = np.r_[lin_reg.intercept_, lin_reg.coef_]

    print("Pesos aprendidos (custom):   ", w_custom)
    print("Pesos aprendidos (sklearn): ", w_sklearn)

    # Verificación
    assert np.allclose(w_custom, w_sklearn, atol=1e-1), f"❌ Los parámetros difieren: {w_custom} vs {w_sklearn}"

    print("✅ Test EJ1 vs scikit-learn superado")

except NotImplementedError:
    print("❌ EJ1 pendiente: implementa gradient_descent")

except AssertionError as e:
    print(str(e))

except Exception as e:
    print(f"⚠️ Error inesperado en test EJ1 sklearn: {e}")



## EJERCICIO 2 — `s_gradient_descent(X, y, learning_rate, n_iterations, w_init)` (autocorregible)

Implementa la función **Stochastic Gradient_descent** (Gradiente por descendiente Estocástico). Debe devolver un vector de pesos. Calcula los pesos por cada ejemplo.


In [ ]:
def s_gradient_descent(X, y, learning_rate=0.1, n_iterations=1000, w_init=None):
   raise NotImplementedError("Implementar la función de gradiente por descenso con regularización L2 (Ridge)")

In [ ]:
# TEST UNITARIO para s_gradient_descent
try:
    import numpy as np
    from sklearn.linear_model import LinearRegression

    # Datos sencillos: y = x (pendiente=1, bias=0)
    X_test_unit = np.array([[1], [2], [3]])
    y_test_unit = np.array([1, 2, 3])

    # Nuestra implementación SGD
    # iniciar semilla y pesos iniciales
    np.random.seed(42)
    initial_w = np.random.randn(2)  # Tamaño 2 para bias + pendiente
    w_sgd = s_gradient_descent(X_test_unit, y_test_unit, learning_rate=0.1, n_iterations=1000, w_init=initial_w)

    # Implementación scikit-learn (solución exacta OLS)
    lin_reg = LinearRegression()
    lin_reg.fit(X_test_unit, y_test_unit)
    w_sklearn = np.r_[lin_reg.intercept_, lin_reg.coef_]

    print("Pesos aprendidos (SGD custom):", w_sgd)
    print("Pesos aprendidos (sklearn):   ", w_sklearn)

    # Verificaciones
    assert w_sgd.shape[0] == 2, "❌ w debe tener tamaño 2 (bias + pendiente)"
    assert np.allclose(w_sgd, w_sklearn, atol=0.2), f"❌ Los parámetros difieren: {w_sgd} vs {w_sklearn}"

    print("✅ Test SGD superado")

except NotImplementedError:
    print("❌ EJ1 pendiente: implementa s_gradient_descent")

except AssertionError as e:
    print(str(e))

except Exception as e:
    print(f"⚠️ Error inesperado en test SGD: {e}")



## EJERCICIO 3 — `mini_batch_gradient_descent(X, y, learning_rate, n_iterations, batch_size, w_init)` (autocorregible)

Implementa la función **Mini Bacth Stochastic Gradient_descent** (Gradiente por descendiente con Mini Grupos). Debe devolver un vector de pesos. Calcula los pesos por cada ejemplo.

In [ ]:
def mini_batch_gradient_descent(X, y, learning_rate=0.1, n_iterations=1000, batch_size=20, w_init=None):
   raise NotImplementedError("Implementar la función de mini-batch gradient descent")

In [ ]:
# TEST UNITARIO Mini-batch GD (con ruido ligero)
try:
    import numpy as np

    np.random.seed(1)

    # Datos: y = 1 + 2x + ruido
    N = 400
    X_test = np.linspace(-5, 5, N).reshape(-1, 1)
    noise = np.random.normal(0, 0.1, size=N)  # ruido pequeño
    y_test = 1 + 2 * X_test.ravel() + noise

    # Iniciar semilla y pesos iniciales
    np.random.seed(42)
    initial_w = np.random.randn(2)  # Tamaño 2 para bias + pendiente
    w_mb = mini_batch_gradient_descent(
        X_test, y_test,
        learning_rate=0.05, n_iterations=2000, batch_size=32, w_init=initial_w
    )

    print("Pesos (Mini-batch, con ruido):", w_mb)

    # Verificaciones
    assert w_mb.shape[0] == 2, "❌ w debe tener tamaño 2 (bias + pendiente)"
    # tolerancias más amplias por el ruido y el carácter estocástico
    assert np.allclose(w_mb, [1.0, 2.0], atol=0.15), f"❌ Parámetros lejos de lo esperado: {w_mb}"

    X_b = np.c_[np.ones((N, 1)), X_test]
    mse = np.mean((X_b @ w_mb - y_test) ** 2)
    assert mse < 0.05, f"❌ MSE demasiado alto: {mse}"

    print("✅ Test Mini-batch (con ruido) superado")

except NotImplementedError:
    print("❌ EJ1 pendiente: implementa mini_batch_gradient_descent")
except AssertionError as e:
    print(str(e))
except Exception as e:
    print(f"⚠️ Error inesperado en test Mini-batch (con ruido): {e}")



## EJERCICIO 4 `GD_momentun(X, y, learning_rate, rho, n_iterations, w_init)` (autocorregible)

Implementa la función ** Gradient_descent mas momentum** (Gradiente por descendiente mas momentum). Debe devolver un vector de pesos. Calcula los pesos por epoch.

In [ ]:
def gd_momentum(X, y, learning_rate=0.1, rho=0.9, n_iterations=1000, w_init=None): # gd con momentum se actualiza por epoch
   raise NotImplementedError("Implementar la función de gradiente por descenso con momentum")
                         

In [ ]:
# TEST UNITARIO para gradient_descent mas momentum
try:
    import numpy as np

    # Datos sencillos: y = x (pendiente=1, bias=0)
    X_test_unit = np.array([[1], [2], [3]])
    y_test_unit = np.array([1, 2, 3])

    # Ejecutar la función
    # iniciar semilla y pesos iniciales
    np.random.seed(42)
    initial_w = np.random.randn(2)  # Tamaño 2 para bias + pendiente
    w_test_unit = gd_momentum(X_test_unit, y_test_unit, learning_rate=0.1, n_iterations=1000, w_init=initial_w)

    print("Pesos aprendidos (test unitario):", w_test_unit)

    # Verificaciones
    assert w_test_unit.shape[0] == 2, "❌ w debe tener tamaño 2 (bias + pendiente)"
    assert np.allclose(w_test_unit, [0, 1], atol=0.1), f"❌ w no es correcto, obtenido {w_test_unit}"

    print("✅ Test EJ1 superado")

except NotImplementedError:
    print("❌ EJ1 pendiente: implementa gd_momentum")

except AssertionError as e:
    print(str(e))

except Exception as e:
    print(f"⚠️ Error inesperado en test EJ1: {e}")
